In [2]:
from monai.networks.nets import UNet
from monai.networks.layers import Norm
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


In [3]:
model = UNet(
    spatial_dims=3,
    in_channels=1,
    out_channels=1,
    channels=(16, 32, 64, 128, 256),
    strides=(2, 2, 2, 2),
    num_res_units=2,
    norm=Norm.BATCH
).to(device)

print(model)

model.load_state_dict(torch.load("best_metric_model.pth"))

UNet(
  (model): Sequential(
    (0): ResidualUnit(
      (conv): Sequential(
        (unit0): Convolution(
          (conv): Conv3d(1, 16, kernel_size=(3, 3, 3), stride=(2, 2, 2), padding=(1, 1, 1))
          (adn): ADN(
            (N): BatchNorm3d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (D): Dropout(p=0.0, inplace=False)
            (A): PReLU(num_parameters=1)
          )
        )
        (unit1): Convolution(
          (conv): Conv3d(16, 16, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
          (adn): ADN(
            (N): BatchNorm3d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (D): Dropout(p=0.0, inplace=False)
            (A): PReLU(num_parameters=1)
          )
        )
      )
      (residual): Conv3d(1, 16, kernel_size=(3, 3, 3), stride=(2, 2, 2), padding=(1, 1, 1))
    )
    (1): SkipConnection(
      (submodule): Sequential(
        (0): ResidualUnit(
          (conv): Sequential(


<All keys matched successfully>

In [4]:
import dataloading.load_atlas as load_atlas

train_id_loader, val_id_loader, test_id_loader = load_atlas.load_atlas()

Found 655 image files
Found 655 mask files
Found 300 test image files
Train data: 524 files
Validation data: 131 files


In [5]:
from monai.inferers import sliding_window_inference
from tqdm import tqdm
import pickle
from pathlib import Path

def compute_id_scores(model, id_loader):
    model.eval()
    id_scores = []
    with torch.no_grad():
        for batch_data in tqdm(id_loader, "Computing ID scores"):
            image = batch_data["image"].to(device)
            output = sliding_window_inference(image, (96, 96, 96), 4, model, overlap=0.5)
            confidence_score = torch.sigmoid(output)
            mcs = torch.max(confidence_score, 1 - confidence_score)
            id_scores.append(torch.min(mcs).cpu().item())
    return id_scores

pickle_filename = "id_scores.pkl"
if Path(pickle_filename).exists():
    with open(pickle_filename, "rb") as fp:
        id_scores = pickle.load(fp)
else:
    id_scores = compute_id_scores(model, val_id_loader)
    with open(pickle_filename, "wb") as fp:
        pickle.dump(id_scores, fp)

id_scores


[0.5002381205558777,
 0.5000490546226501,
 0.5000529289245605,
 0.5000977516174316,
 0.5000520944595337,
 0.5000888109207153,
 0.5007883310317993,
 0.5001527070999146,
 0.5002129077911377,
 0.5000300407409668,
 0.5003063082695007,
 0.500113844871521,
 0.5000293850898743,
 0.500389575958252,
 0.5001211166381836,
 0.5000301599502563,
 0.5000430941581726,
 0.5000050663948059,
 0.500244140625,
 0.5000814199447632,
 0.5002129077911377,
 0.5001280307769775,
 0.50001460313797,
 0.50005042552948,
 0.5000863075256348,
 0.5000125169754028,
 0.5003453493118286,
 0.500045657157898,
 0.5001804828643799,
 0.5000031590461731,
 0.5000084042549133,
 0.5005553960800171,
 0.5002567768096924,
 0.5009927153587341,
 0.5001481175422668,
 0.500002384185791,
 0.5003548860549927,
 0.5037420392036438,
 0.5000173449516296,
 0.5000579953193665,
 0.5001019239425659,
 0.5000748634338379,
 0.5000369548797607,
 0.5000494718551636,
 0.5000289082527161,
 0.5049540996551514,
 0.5002901554107666,
 0.5000149607658386,
 0.5

In [6]:
import numpy as np

def determine_threshold(id_scores, percentile=5):
    threshold = np.percentile(id_scores, percentile)
    return threshold

threshold_90 = determine_threshold(id_scores, percentile=10)
threshold_95 = determine_threshold(id_scores, percentile=5)
threshold_99 = determine_threshold(id_scores, percentile=1)

In [6]:
from dataloading.load_chaos import load_chaos

chaos_loader = load_chaos()

Found 20 Train CT volumes
Found 20 Test CT volumes
Found 0 Train MR volumes
Found 0 Test MR volumes
Total CHAOS volumes: 40
CHAOS dataloader created with 40 samples


In [7]:
id_scores_chaos = compute_id_scores(model, chaos_loader)
id_scores_chaos

Computing ID scores:   0%|          | 0/40 [00:00<?, ?it/s]Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /build/python-pytorch/src/pytorch-cuda/torch/csrc/autograd/python_variable_indexing.cpp:345.)
Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /build/python-pytorch/src/pytorch-cuda/torch/csrc/autograd/python_variable_indexing.cpp:345.)
Computing ID scores: 100%|██████████| 40/40 [01:01<00:00,  1.53s/it]


[0.5000015497207642,
 0.500003457069397,
 0.500004231929779,
 0.500005304813385,
 0.5000030994415283,
 0.5000024437904358,
 0.5000016093254089,
 0.5000004172325134,
 0.5000003576278687,
 0.5000007152557373,
 0.5000006556510925,
 0.5000052452087402,
 0.5000026226043701,
 0.5000010132789612,
 0.5000057816505432,
 0.5000016689300537,
 0.5000030398368835,
 0.5000020861625671,
 0.5000026226043701,
 0.5000027418136597,
 0.500002384185791,
 0.5000016689300537,
 0.5000020265579224,
 0.5000119209289551,
 0.500003457069397,
 0.5000004768371582,
 0.5000009536743164,
 0.5000007748603821,
 0.5000007748603821,
 0.5000023245811462,
 0.5000024437904358,
 0.5000016689300537,
 0.5000003576278687,
 0.5000006556510925,
 0.5000001192092896,
 0.5000001788139343,
 0.5000309944152832,
 0.5000011324882507,
 0.5000016093254089,
 0.5000011920928955]

In [8]:
val_id_scores = np.array(compute_id_scores(model, test_id_loader))
id_scores = np.concatenate([np.array(id_scores_chaos), val_id_scores])
labels = np.concatenate([np.ones_like(id_scores_chaos), np.zeros_like(val_id_scores)])

Computing ID scores: 100%|██████████| 300/300 [01:52<00:00,  2.67it/s]


In [9]:
from sklearn.metrics import average_precision_score
from metrics import fpr

anomaly_scores = -id_scores

aupr = average_precision_score(labels, anomaly_scores)
fpr90 = fpr(labels, anomaly_scores, percentile=10)
fpr95 = fpr(labels, anomaly_scores, percentile=5)
fpr99 = fpr(labels, anomaly_scores, percentile=1)

print(f"AUPR: {aupr:.4f}, fpr90: {fpr90:.4f}, fpr95: {fpr95:.4f}, fpr99: {fpr99:.4f}")

AUPR: 0.6162, fpr90: 0.0800, fpr95: 0.0867, fpr99: 0.3033


In [7]:
from dataloading.load_brats import load_brats
brats_loader = load_brats()
len(brats_loader)

Found 585 Train BRATS volumes
Found 87 Test BRATS volumes


672

In [8]:
id_scores_brats = compute_id_scores(model, brats_loader)
id_scores_brats

Computing ID scores:   0%|          | 0/672 [00:00<?, ?it/s]WARNING: In /work/ITK-source/ITK/Modules/IO/ImageBase/include/itkImageSeriesReader.hxx, line 478
ImageSeriesReader (0x555c52a826b0): Non uniform sampling or missing slices detected,  maximum nonuniformity:0.00101728

ImageSeriesReader (0x555c52b4ba80): Non uniform sampling or missing slices detected,  maximum nonuniformity:0.000545123

Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /build/python-pytorch/src/pytorch-cuda/torch/csrc/autograd/python_variable_indexing.cpp:345.)
Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, 

[0.500013530254364,
 0.5000088214874268,
 0.5000184178352356,
 0.5000823736190796,
 0.5000578165054321,
 0.5001301169395447,
 0.5007257461547852,
 0.5001540780067444,
 0.5000288486480713,
 0.500036358833313,
 0.5000735521316528,
 0.5001437664031982,
 0.5000144243240356,
 0.50001460313797,
 0.5001677870750427,
 0.5000514388084412,
 0.500025749206543,
 0.50002121925354,
 0.500278651714325,
 0.5000127553939819,
 0.5001085996627808,
 0.5004251003265381,
 0.5000222325325012,
 0.5000995993614197,
 0.5000646710395813,
 0.500067949295044,
 0.5000331997871399,
 0.5001619458198547,
 0.5000219941139221,
 0.5002368092536926,
 0.5000472068786621,
 0.5000662803649902,
 0.5003964900970459,
 0.5000001788139343,
 0.5003005266189575,
 0.5000302791595459,
 0.5000045299530029,
 0.5000896453857422,
 0.50007164478302,
 0.5001316070556641,
 0.5000842809677124,
 0.500234067440033,
 0.5000368356704712,
 0.5000556707382202,
 0.5000122785568237,
 0.5001145601272583,
 0.5003310441970825,
 0.5002323985099792,
 0.5

In [9]:
val_id_scores = np.array(compute_id_scores(model, test_id_loader))
id_scores = np.concatenate([np.array(id_scores_brats), val_id_scores])
labels = np.concatenate([np.ones_like(id_scores_brats), np.zeros_like(val_id_scores)])

Computing ID scores:   0%|          | 0/300 [00:00<?, ?it/s]Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /build/python-pytorch/src/pytorch-cuda/torch/csrc/autograd/python_variable_indexing.cpp:345.)
Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /build/python-pytorch/src/pytorch-cuda/torch/csrc/autograd/python_variable_indexing.cpp:345.)
Computing ID scores: 100%|██████████| 300/300 [01:58<00:00,  2.54it/s]


In [10]:
from sklearn.metrics import average_precision_score
from metrics import fpr

anomaly_scores = -id_scores

aupr = average_precision_score(labels, anomaly_scores)
fpr90 = fpr(labels, anomaly_scores, percentile=10)
fpr95 = fpr(labels, anomaly_scores, percentile=5)
fpr99 = fpr(labels, anomaly_scores, percentile=1)

print(f"AUPR: {aupr:.4f}, fpr90: {fpr90:.4f}, fpr95: {fpr95:.4f}, fpr99: {fpr99:.4f}")

AUPR: 0.7176, fpr90: 0.7800, fpr95: 0.8433, fpr99: 0.9400


In [11]:
from sklearn.metrics import accuracy_score

pred = id_scores < threshold_95

accuracy_score(labels, pred)


0.345679012345679